# SOXX 对 QQQ 下跌 beta 缺口与现金转 SOXL 局部价值

冻结一个不同于已关闭H5尾部任务的费用后局部接收目标；只做必要排序，保留来源时点限制，不拟合或跑完整账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
root=Path.cwd();card=root/'research/stage1_qqq_downside_beta_soxl_local_rank_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
f=root/s['factor_card'];a=root/s['label_card']
assert sha(f/'summary.json')==s['factor_summary_sha256'] and sha(f/'plan.json')==s['factor_plan_sha256']
assert sha(a/'summary.json')==s['label_summary_sha256'] and sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sf=json.loads((f/'summary.json').read_text());sa=json.loads((a/'summary.json').read_text())
assert sf['nas_readback']['verified'] and sf['nas_readback']['sha256']==s['factor_NAS_sha256']
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['label_NAS_sha256']
assert sha(f/'qlib_recorder'/s['factor_experiment_id']/s['factor_recorder_id']/'artifacts/inputs/dataset')==s['factor_dataset_sha256']
assert sha(f/'qlib_recorder'/s['factor_experiment_id']/s['factor_recorder_id']/'artifacts/analysis/source_frame')==s['factor_frame_sha256']
assert sha(a/'qlib_recorder'/s['label_experiment_id']/s['label_recorder_id']/'artifacts/analysis/labels')==s['label_artifact_sha256']
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['factor_experiment_id'],recorder_id=s['factor_recorder_id']);dataset=rec.load_object('inputs/dataset');frame=rec.load_object('analysis/source_frame')
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['label_experiment_id'],recorder_id=s['label_recorder_id']);labels=rec.load_object('analysis/labels')
prepared=dataset.prepare('train',col_set=['feature','meta'])
assert len(prepared)==3020 and len(labels)==1061
train_dates=pd.DatetimeIndex(prepared.index.get_level_values('datetime'))
assert train_dates.is_unique and train_dates.equals(pd.DatetimeIndex(frame.loc['2011-01-03':'2022-12-30'].index))
np.testing.assert_allclose(prepared[('feature','downside_beta_gap_60')].to_numpy(float),frame.loc[train_dates,'downside_beta_gap_60'].to_numpy(float),equal_nan=True,atol=0,rtol=0)
signal=pd.DatetimeIndex(labels.signal_date);trade=pd.DatetimeIndex(labels.trade_date)
assert signal.is_unique and trade.is_unique and signal.isin(train_dates).all()
assert signal.isin(pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2022-12-30',freq='day'))).all()
assert (signal<trade).all() and labels.soxl_local_fee_net_advantage.notna().sum()==688
assert (pd.to_datetime(labels.loc[labels.soxl_local_fee_net_advantage.notna(),'label_available_at'])>pd.to_datetime(labels.loc[labels.soxl_local_fee_net_advantage.notna(),'decision_at'])).all()
score=-frame.loc[signal,'downside_beta_gap_60'].to_numpy(float)
assert np.isfinite(score).all()
x=pd.DataFrame({'signal_date':signal,'trade_date':trade,'year':trade.year,'quarter':trade.quarter,
                'cap_binding':labels.cap_binding.to_numpy(bool),'eligible':labels.eligible.to_numpy(bool),
                'score':score,'label':labels.soxl_local_fee_net_advantage.to_numpy(float)})
assert x.loc[x.label.notna(),'eligible'].all()
def rank(g):
 k=g[g.label.notna()].copy();z=k[k.label.abs()>1e-12];y=(z.label>0).to_numpy(int)
 auc=float(roc_auc_score(y,z.score.to_numpy(float))) if len(set(y))==2 else None
 sp=float(spearmanr(k.score.to_numpy(float),k.label.to_numpy(float)).statistic) if len(k)>2 and k.score.nunique()>1 and k.label.nunique()>1 else None
 return {'sessions':len(g),'mature':len(k),'positive':int((k.label>1e-12).sum()),'negative':int((k.label<-1e-12).sum()),
         'zero':int((k.label.abs()<=1e-12).sum()),'auc':auc,'spearman':sp,'mean_label':float(k.label.mean()) if len(k) else None}
full=rank(x)
annual=[{'year':int(year),'partial_endpoint':year==2018,**rank(g)} for year,g in x.groupby('year',sort=True)]
quarter=[{'year':int(year),'quarter':int(q),**rank(g)} for (year,q),g in x.groupby(['year','quarter'],sort=True)]
by_binding={str(binding).lower():rank(g) for binding,g in x.groupby('cap_binding')}
binding_era=[{'era':'2020_2021',**rank(x[x.year.isin([2020,2021])&x.cap_binding])},{'era':'2022',**rank(x[(x.year==2022)&x.cap_binding])}]
assert len(annual)==5 and len(quarter)==17 and full['mature']==688 and sum(a['mature'] for a in annual)==688
required=[next(z for z in annual if z['year']==y) for y in (2020,2021,2022)]
gates={'each_2020_2022_auc_above_half':all(z['auc'] is not None and z['auc']>.5 for z in required),
       'each_2020_2022_spearman_positive':all(z['spearman'] is not None and z['spearman']>0 for z in required),
       'binding_era_spearman_positive':all(z['spearman'] is not None and z['spearman']>0 for z in binding_era)}
result={'status':'read_only_QQQ_downside_beta_SOXL_fee_net_local_value_necessary_rank','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'factor':s['factor_recorder_id'],'labels':s['label_recorder_id']},
        'full_sessions':len(x),'mature_labels':full['mature'],'full':full,'annual':annual,'quarter':quarter,'by_binding':by_binding,'binding_era':binding_era,
        'gates':gates,'necessary_gate_pass':all(gates.values()),'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Current Yahoo historical source does not certify first public market-data receipt or revisions','2018-2022 local labels are reused and do not prove independent validation','Same-next-target local value is not the exact f696 own-state complete funded SOXX account']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'full':full,'years':[(z['year'],z['auc'],z['spearman']) for z in annual],'binding_era':binding_era,'gates':gates})


{'full': {'sessions': 1061, 'mature': 688, 'positive': 345, 'negative': 311, 'zero': 32, 'auc': 0.4731254951302484, 'spearman': -0.025596024614003485, 'mean_label': 4.957468748976795e-05}, 'years': [(2018, 0.6818181818181818, 0.03333333333333333), (2019, 0.40816326530612246, -0.13504454574052757), (2020, 0.5329793751276292, 0.08940591875524585), (2021, 0.513136288998358, -0.04375023574150672), (2022, 0.4742075408742075, -0.07725594625673304)], 'binding_era': [{'era': '2020_2021', 'sessions': 238, 'mature': 238, 'positive': 139, 'negative': 99, 'zero': 0, 'auc': 0.5091199767458761, 'spearman': 0.02959847145256069, 'mean_label': 0.0002783639726661111}, {'era': '2022', 'sessions': 246, 'mature': 245, 'positive': 111, 'negative': 133, 'zero': 1, 'auc': 0.4705005757637336, 'spearman': -0.07756407641025224, 'mean_label': -0.00016301961900291737}], 'gates': {'each_2020_2022_auc_above_half': False, 'each_2020_2022_spearman_positive': False, 'binding_era_spearman_positive': False}}


## 中文结论

# SOXX/QQQ下跌 beta 缺口未能排序 SOXL 局部部署价值

固定使用原Qlib信号日前60个完整交易日的 `SOXX对QQQ下跌日beta−全日beta`，事前方向为负：额外下跌敏感度越大，预测现金转SOXL的下一日0bp Tiered含费局部价值越低。来源卡的3020个训练日 DatasetH、原件哈希、NAS定向回读及1061日原架构局部标签逐件核对；688个成熟动作全保留，部分2018、完整2019–2022、17季度和受限/未受限层均计算。0新行情、拟合、资金账户或Recorder。

固定分数的全段正/负局部标签AUC **0.4731**，连续局部价值Spearman **−0.0256**。完整2020/2021/2022年AUC **0.5330/0.5131/0.4742**，Spearman **+0.0894/−0.0438/−0.0773**；受限层2020/2021合并与2022分别为 **+0.0296/−0.0776**。预定三项必要门全部失败。个别好年份不能用来选择正负号、调整60日窗口或筛风险状态。

决定 `not_promoted_fixed_QQQ_downside_beta_SOXL_local_value_rank_failed`，不进入增量回归或精确f696自身状态0bp账户。该接收目标与此前已关闭的五日SOXX尾部标签不同，但这次费用后局部价值也没有稳定方向。当前Yahoo修订历史没有认证当年首次收到的原始字节；即使排序通过，也仍须解决来源时点、完整含费SOXX硬门、2/5bp与独立前向。现有结果均为复用训练历史的开发反馈，SOXX目标未达成。
